## 그룹과제-3: ML Pipeline 구축 (모델링·앙상블)

`01_feature_engineering.ipynb`에서 만든 피처를 바탕으로 **OOF·Voting·가중 앙상블**까지 수행하는 노트북이다. 목적·데이터·평가 기준은 01과 동일하다.

In [ ]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np
#!pip install category_encoders
import category_encoders as ce

# Visualization
import matplotlib.pylab as plt
from matplotlib import font_manager, rc

font_path = "C:/Windows/Fonts/NGULIM.TTF"
font = font_manager.FontProperties(fname=font_path).get_name()
rc('font', family=font)
import seaborn as sns
%matplotlib inline

# Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectPercentile

# Hyperparameter Optimization
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Modeling
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.base import ClassifierMixin
from sklearn.base import BaseEstimator, TransformerMixin

# Evaluation
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import log_loss

# Utility
import os
import time
import datetime
import random
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image
import pickle
from tqdm import tqdm
import platform
from itertools import combinations
from scipy.stats.mstats import gmean

#추가한것
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn import set_config
from optuna.distributions import CategoricalDistribution, IntDistribution, FloatDistribution
from optuna.integration import OptunaSearchCV, ShapleyImportanceEvaluator
import re
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from category_encoders import TargetEncoder, BinaryEncoder
import optuna
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
##코드끝나면 울리는 코드
import winsound as sd
def beepsound():
    fr = 2000    # range : 37 ~ 32767
    du = 1000     # 1000 ms ==1second
    sd.Beep(fr, du) # winsound.Beep(frequency, duration)
# !pip install bayesian-optimization
# from bayes_opt import BayesianOptimization
#!pip install num2words
# from num2words import num2words

# 1. Load data

In [ ]:
X_train = pd.read_csv("../data/X_train.csv", encoding='cp949')
y_train = pd.read_csv('../data/y_train.csv', encoding='cp949').Salary  #ID, Salary

X_test = pd.read_csv("../data/X_test.csv", encoding='euc-kr')

In [ ]:
X_train.info() # "직무태그,근무형태, 어학시험,대학성적" 결측치 존재

In [ ]:
X_test.info()

In [ ]:
sns.distplot(y_train); plt.show() 

---
# 2. Preprocessing

## **1) 결측치 처리**

In [ ]:
# 1.(cat) 직무태그 -> 세부직종 값으로 대체  
X_train['직무태그'] = X_train['직무태그'].fillna('없음')
X_test['직무태그'] = X_test['직무태그'].fillna('없음')

# 2.(cat) 근무형태 -> 최빈값
X_train['근무형태'] = X_train['근무형태'].fillna(X_train['근무형태'].mode()[0])                               
X_test['근무형태'] = X_test['근무형태'].fillna(X_test['근무형태'].mode()[0])      

# 3.(cat) 어학시험 -> '없음' (0이 아닌 cat형으로 바꿔야 함)
X_train['어학시험'] = X_train['어학시험'].fillna('없음')
X_test['어학시험'] = X_test['어학시험'].fillna('없음')

# 4.(num) 대학성적 -> 최빈값
X_train['대학성적'] = X_train['대학성적'].fillna(X_train['대학성적'].mode()[0]).astype(int)
X_test['대학성적'] = X_test['대학성적'].fillna(X_test['대학성적'].mode()[0]).astype(int)

display(X_train.head(), X_test.head())

## **2) Feature Engineering**

> ### **직종, 세부직종**

1. 직종별 세부직종에서 분포가 25% 이하인 값 -> '{해당 직종 이름} + 기타'로 대체
2. 직종의 고유값들을 새로운 컬럼으로 만들고, 해당 컬럼들의 값으로 해당 직종에 해당하는 행의 세부직종 값으로 매핑

=> 두 기능을 하나의 사용자 정의 함수로 구현

In [ ]:
def occupation(df):
    ### 1. 25% 이하인 값을 '{해당 직종 이름} + 기타'로 대체
    for i in df['직종'].unique():
        detail = df[df['직종'] == i]['세부직종'].value_counts()
        selected_detail = detail[detail >= detail.quantile(0.25)].index.tolist()
        df.loc[(df['직종'] == i) & (~df['세부직종'].isin(selected_detail)), '세부직종'] = i + ' 기타'
    
    ### 2. '직종'의 고유값들을 가져와서 새로운 컬럼 생성
    unique_jobs = df['직종'].unique()
    for job in unique_jobs:
        df[job] = '-'  
    for i, row in df.iterrows():
        df.loc[i, row['직종']] = row['세부직종']   # 해당 직종의 세부직종 값을 해당하는 컬럼에 매핑
    
    ### 3. 세부직종_종류_개수
    df['세부직종_종류_개수'] = df['세부직종'].apply(lambda x: len(x.split('·')))

In [ ]:
occupation(X_train)
occupation(X_test)

> ### **직무태그**

> ### **근무형태**

In [ ]:
X_train['근무형태'] = np.where(X_train.근무형태 == "정규직", '정규직', '비정규직')
X_test['근무형태'] = np.where(X_test.근무형태 == "정규직", '정규직', '비정규직')


# 정규직: 1, 비정규직: 0으로 변환 & dtype int로 변환
X_train['근무형태'] = X_train['근무형태'].apply(lambda x: 1 if x == '정규직' else 0).astype(int)
X_test['근무형태'] = X_test['근무형태'].apply(lambda x: 1 if x == '정규직' else 0).astype(int)

> ### **근무지역**

In [ ]:
# 지역 컬럼 + 해외 컬럼 만들기

def clean_location(df):
    unique_locations = df['근무지역'].str.split(',').explode().str.strip().unique()
    countries = ['일본', '인도', '미국', '해외', '캐나다', '말레이시아', '중국', '홍콩',
                    '인도네시아', '대만', '싱가포르', '방글라데시', '프랑스', '필리핀', '러시아']
    df['해외'] = 0
    for location in unique_locations:
        if location.strip() and location not in countries:  # 공백이 아니고 countries 내에 없는 경우
            df[location] = df['근무지역'].apply(lambda x: 1 if location in x else 0)
        elif location.strip():  # 공백이 아니면서 countries 내에 있는 경우
            df.loc[df['근무지역'].str.contains(location, na=False), '해외'] = 1

In [ ]:
clean_location(X_train)
clean_location(X_test)

In [ ]:
countries = X_train.columns[X_train.columns.get_loc('해외'):].to_list()
countries

In [ ]:
def filter_columns_by_ones_count(X_train, countries):
    # 각 피처의 1의 개수 계산
    ones_count = {feature: X_train[feature].sum() for feature in countries}

    # 1의 개수를 기준으로 내림차순 정렬
    sorted_ones_count = sorted(ones_count.items(), key=lambda x: x[1], reverse=True)

    # X_train.shape[0] * 0.05 보다 큰 1의 개수를 가진 열들을 필터링
    filtered_columns = [feature for feature, count in sorted_ones_count if count > X_train.shape[0] * 0.05]

    return filtered_columns

filtered_columns = filter_columns_by_ones_count(X_train, countries)
filtered_columns #얘네만 살려

> ### **출신대학**

In [ ]:
# 방법1. 사용자 정의 함수
def 상위10_대학(df):
    top10_list = ['연세대학교', '성균관대학교', '중앙대학교', '이화여자대학교', '서울과기대학교', '세종대학교', '성신여자대학교', '동덕여자데학교', '인천대학교', '서울여자대학교']
    df['상위10_대학'] = 0  
    for i, 대학 in enumerate(df['출신대학']):
        if 대학 in top10_list:
            df.at[i, '상위10_대학'] = 1  # 해당 행의 '상위10_대학' 값을 1로 설정


# 방법2. 사용자 정의 함수
'''
top_list=['연세대학교','성균관대학교', '중앙대학교','이화여자대학교','서울과기대학교','세종대학교','성신여자대학교','동덕여자데학교','인천대학교','서울여자대학교']
X_train['상위10_대학'] = X_train['출신대학'].apply(lambda x: 1 if x in top_list else 0)
X_test['상위10_대학'] = X_test['출신대학'].apply(lambda x: 1 if x in top_list else 0)'''

In [ ]:
상위10_대학(X_train)
상위10_대학(X_test)

>### **대학전공**

In [ ]:
def major(df):
    df['대학전공'] = df['대학전공'].str.replace(r'[^a-zA-Z0-9가-힣]+|(전공|과|부)$', '', regex=True).str.lower()
    df['대학전공_대분류'] = df['대학전공'].apply(lambda x: x[:2])

In [ ]:
major(X_train)
major(X_test)

> ### **어학시험**

In [ ]:
X_train['어학시험'] = X_train['어학시험'].replace(' ', '없음')
X_test['어학시험'] = X_test['어학시험'].replace(' ', '없음')

> ### **자격증**

In [ ]:
X_train['자격증'] = X_train['자격증'].replace({'無': 0, '有': 1})
X_test['자격증'] = X_test['자격증'].replace({'無': 0, '有': 1})

X_train['자격증'].value_counts()

In [ ]:
''' 최종 X_train & 최종 X_test '''
print(X_train.columns, len(X_train.columns))
print(X_test.columns, len(X_test.columns))

---
# 3. 피처 생성

In [ ]:
### '근무경력' 컬럼 수치형으로 변환
def convert_to_num(df):
    df['근무경력'] = df['근무경력'].apply(lambda x: int(x.split('년')[0])*12 + int(x.split(' ')[1].replace('개월', '')) 
                                                    if '년' in x else int(x.replace('개월', ''))                )
convert_to_num(X_train)
convert_to_num(X_test)

In [ ]:
## 1. 대학성적_근무경력 -> num
X_train['대학성적_근무경력'] = X_train['대학성적'] + X_train['근무경력']
X_test['대학성적_근무경력'] = X_test['대학성적'] + X_test['근무경력']

In [ ]:
## 2. 직종_근무형태
X_train['직종_근무형태'] = X_train['직종'] + '_' + X_train['근무형태'].astype(str)
X_test['직종_근무형태'] = X_test['직종'] + '_' + X_test['근무형태'].astype(str)

In [ ]:
## 3. 자격증_근무형태 
X_train['자격증_근무형태'] = X_train['자격증'] + X_train['근무형태']
X_test['자격증_근무형태'] = X_test['자격증'] + X_test['근무형태']

In [ ]:
## 4. 대학성적_근무형태
X_train['대학성적_근무형태'] = X_train['대학성적'] * X_train['근무형태']
X_test['대학성적_근무형태'] = X_test['대학성적'] * X_test['근무형태']

In [ ]:
## 5. 출신대학_전공
X_train['출신대학_전공'] = X_train['출신대학'] + X_train['대학전공']
X_test['출신대학_전공'] = X_test['출신대학'] + X_test['대학전공']

In [ ]:
## 6. 상위10대학_근무형태
X_train['상위10대학_근무형태'] = X_train['상위10_대학'] + X_train['근무형태']
X_test['상위10대학_근무형태'] = X_test['상위10_대학'] + X_test['근무형태']

In [ ]:
## 7. 세부직종개수_대학성적
X_train['세부직종개수_대학성적'] = X_train['세부직종_종류_개수'] * X_train['대학성적']
X_test['세부직종개수_대학성적'] = X_test['세부직종_종류_개수'] * X_test['대학성적']

---
# 4. 수치형/범주형 피처 분리 & 학습/평가 데이터 분할

In [ ]:
display(X_train.head(), X_test.head())

In [ ]:
''' 최종 X_train & 최종 X_test '''
print(X_train.columns, len(X_train.columns))
print(X_test.columns, len(X_test.columns))

In [ ]:
print(X_train.info())
print(X_test.info())

In [ ]:
def get_value_counts(column_name):
    print(f"---------- {column_name} ----------")
    unique_values = X_train[column_name].value_counts()
    
    print(f"**고유값 개수** : {X_train[column_name].nunique()}개")
    if len(unique_values) < 10:
        print(unique_values)
        print()
    else:
        print(unique_values.head(10))
        print()

In [ ]:
get_value_counts('상위10대학_근무형태')

In [ ]:
col_num =['ID', '직종', '세부직종', '직무태그', '근무경력', '근무형태', '근무지역', '출신대학', '대학전공',
       '어학시험', '자격증', '대학성적', '문화·예술·신문·방송', '경영·기획·회계·사무', 'IT·게임',
       '영업·판매·TM', '기술·과학·산업', '재료·화학·섬유·의복', '전문·교육·자격', '건설·기계·전기·전자', '디자인',
       '통신·모바일', '기타 직종', '세부직종_종류_개수', '해외', '서울', '경기', '부산', '충북', '전국',
       '전북', '인천', '대전', '기타', '경남', '광주', '충남', '강원', '전남', '울산', '경북', '제주',
       '대구', '상위10_대학', '대학전공_대분류', '대학성적_근무경력', '직종_근무형태', '자격증_근무형태',
       '대학성적_근무형태', '출신대학_전공', '상위10대학_근무형태', '세부직종개수_대학성적']

len(col_num) - 14  #ID, 지역 13개

In [ ]:
### 수치형/범주형 피처 분리 (& 학습/평가 데이터 분할)

numeric_features =     ['근무경력', '대학성적', '세부직종_종류_개수', '대학성적_근무경력', '자격증_근무형태', '세부직종개수_대학성적',
                        '대학성적_근무형태', '상위10대학_근무형태'] 

categorical_features1 = ['직종','출신대학', '어학시험', '대학전공_대분류', '직종_근무형태', '출신대학_전공', 
                        '문화·예술·신문·방송', '경영·기획·회계·사무', 'IT·게임', '영업·판매·TM', '기술·과학·산업', '재료·화학·섬유·의복', '전문·교육·자격',
                        '건설·기계·전기·전자', '디자인', '통신·모바일', '기타 직종']  #라벨
categorical_features2 = ['세부직종', '직무태그', '근무지역', '대학전공']  #캣부스트

binary_features =      ['근무형태', '자격증', '상위10_대학', '해외', '서울', '경기', '부산', '인천', '대전']  #원핫
# binary_features pca축소할 애들 뽑음

X_train = X_train[numeric_features + categorical_features1 + categorical_features2 + binary_features]   #+binary_features]  # 순서 주의!!!
X_test = X_test[numeric_features + categorical_features1 + categorical_features2 + binary_features]     #+binary_features]

X_train.shape, X_test.shape

In [ ]:
X_test.iloc[:, 25:29]

---
# 4. 파이프라인 구축

In [ ]:
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold

In [ ]:
### 이상치 처리

def remove_outlier(X, q=0.05):  
    df = pd.DataFrame(X)
    return df.apply(lambda x: x.clip(x.quantile(q), x.quantile(1-q)), axis=0).values

####  **1) 파이프라인 구축**
수치형과 범주형 피처를 다르게 처리할 수 있는 ColumnTransformer를 활용

In [ ]:
# numeric_features + categorical_features1 + categorical_features2 + binary_features
from category_encoders  import CatBoostEncoder
from sklearn.preprocessing import LabelEncoder
from category_encoders import BinaryEncoder
from category_encoders import CountEncoder
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("outlier", FunctionTransformer(remove_outlier, kw_args={'q':0.05})), # 함수를 전처리기로 변환하여 sklearn에 없는 새로운 전처리기를 만듬
        ("scaler", StandardScaler()), #PowerTransformer / StandardScaler
    ]
)

categorical_transformer1 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder", OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1)),
    ]
)


categorical_transformer2 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # 결측치를 해당 열의 가장 빈번한 값으로 대체
        ("encoder", CountEncoder()),  # CountEncoder를 사용한 인코딩
        ("scaler", MinMaxScaler())  # MinMaxScaler를 사용한 스케일링
    ]
)


binary_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=True)),   
    ]
)
'''("corpus", FunctionTransformer(lambda x: x.str.replace('·',',').str.split(',').str.join(" "))),
           ("BoW", CountVectorizer()),'''  


# 피처별 각각 fit, 튜닝
column_transformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer,       numeric_features),
        ("cat1", categorical_transformer1,  categorical_features1),
        ("cat2", categorical_transformer2, categorical_features2),
        ("bin", binary_transformer,        binary_features)         
    ]
)

##=======================================================================================================

## 전처리 파이프라인 구성
preprocessor = Pipeline(
    steps=[
        ("column", column_transformer),
        # ("selector", SelectPercentile(percentile=50)),
    ]
)


set_config(display="diagram")
preprocessor

In [ ]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)

In [ ]:
X_train = preprocessor.fit_transform(X_train, y_train)
X_test = preprocessor.transform(X_test)

In [ ]:
X_train

In [ ]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)

In [ ]:
X_train

In [ ]:
CATBOOST_VERSION = 1.0
NFOLDS = 5
SEED = 0
Q = 0.05

In [ ]:
%%time

## catboost

# 최적화된 하이퍼파라미터로 OOF를 수행하여 최종 CatBoost 모형 생성:
# No tuning => tuning한 모델에 비해 성능이 떨어지지 않음
#다실험해봐야댐 ㅇㅇ, 
#sscv = ShuffleSplit(test_size=.3334, n_splits=5, random_state=0)
models = cross_validate(CatBoostRegressor(iterations=10000,eval_metric = 'RMSE', depth = 5, 
                                          early_stopping_rounds = 1000,learning_rate = 0.1, verbose = 200),
                        X_train, y_train, 
                        cv=NFOLDS, 
                        scoring='neg_mean_squared_error', 
                        return_estimator=True)
oof_pred = np.array([m.predict(X_test) for m in models['estimator']]).mean(axis=0)

scores = models['test_score']
print("\nCatBoost CV scores: ", np.sqrt(-1*scores))
print("CatBoost CV mean = %.2f" % np.sqrt(-1*scores.mean()), "with std = %.2f" % np.sqrt(scores.std()))

In [ ]:
#LGBM
#LGB
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_validate
import numpy as np

models = cross_validate(LGBMRegressor(num_estimators=10000),
                        X_train, y_train,
                        cv=NFOLDS,
                        scoring='neg_mean_squared_error',
                        return_estimator=True,
                        fit_params={ "eval_metric": "rmse", "verbose": 200})

oof_pred = np.array([m.predict(X_test) for m in models['estimator']]).mean(axis=0)
("\nLightGBM CV scores: ", np.sqrt(-1 * scores))
print("LightGBM CV mean = %.2f" % np.sqrt(-1 * scores.mean()), "with std = %.2f" % np.sqrt(scores.std()))

scores = models['test_score']


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_validate
import numpy as np

# Your data and NFOLDS definition
# ...

# Define early stopping
early_stopping_rounds = 100  # Set an appropriate value

models = cross_validate(XGBRegressor(learning_rate=0.1, max_depth=5, n_estimators=10000),
                        X_train, y_train,
                        cv=NFOLDS,
                        scoring='neg_mean_squared_error',
                        return_estimator=True,
                        fit_params={
                            "eval_metric": "rmse",
                            "early_stopping_rounds": early_stopping_rounds,
                            "eval_set": [(X_train, y_train)]  # Evaluation set
                        })

oof_pred = np.array([m.predict(X_test) for m in models['estimator']]).mean(axis=0)

scores = models['test_score']
print("\nXGBoost CV scores: ", np.sqrt(-1 * scores))
print("XGBoost CV mean = %.2f" % np.sqrt(-1 * scores.mean()), "with std = %.2f" % np.sqrt(scores.std()))


In [ ]:
#RandomForest
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

# Create an imputer object with mean strategy
imputer = SimpleImputer(strategy='mean')

# Fit the imputer to X_train and transform it
X_train_imputed = imputer.fit_transform(X_train)

# Now, use RandomForestRegressor with the imputed data
random_forest = RandomForestRegressor(n_estimators=10000, max_depth=5)
models = cross_validate(random_forest,
                        X_train_imputed, y_train,
                        cv=NFOLDS,
                        scoring='neg_mean_squared_error',
                        return_estimator=True)
oof_pred = np.array([m.predict(X_test) for m in models['estimator']]).mean(axis=0)

scores = models['test_score']
print("\nRandomForest CV scores: ", np.sqrt(-1 * scores))
print("RandomForest CV mean = %.2f" % np.sqrt(-1 * scores.mean()), "with std = %.2f" % np.sqrt(scores.std()))

In [ ]:
# #DT
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.model_selection import cross_validate, ShuffleSplit
# import numpy as np

# # Your data and NFOLDS definition
# # ...

# # sscv = ShuffleSplit(test_size=0.3334, n_splits=5, random_state=0)

# # Define the DecisionTreeRegressor model with max_depth
# decision_tree = DecisionTreeRegressor(max_depth=15)  # Set max_depth here

# models = cross_validate(decision_tree,
#                         X_train, y_train,
#                         cv=NFOLDS,
#                         scoring='neg_mean_squared_error',
#                         return_estimator=True)

# oof_pred = np.array([m.predict(X_test) for m in models['estimator']]).mean(axis=0)

# scores = models['test_score']
# print("\nDecisionTreeRegressor CV scores: ", np.sqrt(-1 * scores))
# print("DecisionTreeRegressor CV mean = %.2f" % np.sqrt(-1 * scores.mean()), "with std = %.2f" % np.sqrt(scores.std()))


In [ ]:
import winsound as sd
def beepsound():
    fr = 2000    # range : 37 ~ 32767
    du = 1000     # 1000 ms ==1second
    sd.Beep(fr, du) # winsound.Beep(frequency, duration)
beepsound()

####  **2)** 파이프라인을 통한 **모형 학습** 및 **성능 확인**


In [ ]:
''' 이번 과제 핵심 포인트 '''

# 노트북 나가면 사라지니까
# 튜닝한 모델을 피클로 저장해서
# 피클을 불러오고, STACKING 하기

#최대한 파이프라인 활용하고, 모델링을 최대한 깔끔하게!

In [ ]:
# submission 화일 생성
filename = f'catboost_{CATBOOST_VERSION}_{scores.mean():.2f}.csv'
pd.DataFrame({'ID':test_id, 'Salary':oof_pred}).to_csv(filename, index=False)